<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week9_day5_mini_projet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cellule 1 — Installation des dépendances


In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "fastmcp>=2.0.0" \
  "mcp-server-git" \
  "nest_asyncio"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.2/765.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.4/232.4 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.0/273.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Cellule 2 — Vérifier / installer Node.js (requis pour les serveurs MCP via npx)


In [ ]:
!node --version 2>/dev/null || (apt-get -qq update && apt-get -qq install -y nodejs npm)
!node --version
!npx --version

v20.19.0
v20.19.0
10.8.2


Cellule 3 — Configurer la clé API Gemini


In [ ]:
import os
from google.colab import userdata

# Mise à jour avec la nouvelle clé du compte
# Assurez-vous d'avoir mis à jour 'GOOGLE_API_KEY' dans vos Secrets Colab
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("✓ GOOGLE_API_KEY mise à jour avec le nouveau compte")

✓ GOOGLE_API_KEY mise à jour avec le nouveau compte


Cellule 4 — Créer le workspace de recherche + initialiser Git


In [ ]:
import subprocess, textwrap
from pathlib import Path

WORKDIR = Path("/content/research_workspace")
WORKDIR.mkdir(exist_ok=True)

# Init git repo
for cmd in [
    ["git", "init"],
    ["git", "config", "user.email", "assistant@research.ai"],
    ["git", "config", "user.name", "Research Assistant"],
    ["git", "config", "commit.gpgsign", "false"],
]:
    subprocess.run(cmd, cwd=WORKDIR, check=True, capture_output=True)

# Fichier : notes de recherche
(WORKDIR / "paper_notes.md").write_text(textwrap.dedent("""\
    # Research Notes: Agentic LLMs

    ## Key Citations
    - Smith et al. (2023). "Agentic LLMs." Nature AI.
    - Jones & Lee (2022). "Tool Use in LLMs." ACL.
    - Brown et al. (2020). "Language Models are Few-Shot Learners." NeurIPS.

    ## Abstract
    This paper investigates the use of large language models as autonomous agents
    that can use tools. We show that combining planning with tool-use improves
    performance on complex tasks.

    ## TODO
    - Add more citations
    - Write related work section
"""))

# Fichier : BibTeX incomplet (pour tester validate_bibtex)
(WORKDIR / "references.bib").write_text(textwrap.dedent("""\
    @article{smith2023,
        title = {Agentic LLMs},
        author = {Smith, John and Doe, Jane},
        year = {2023}
    }
    @inproceedings{jones2022,
        title = {Tool Use in LLMs},
        author = {Jones, Alex and Lee, Maria},
        year = {2022}
    }
    @misc{brown2020,
        title = {Language Models are Few-Shot Learners},
        author = {Brown, Tom},
        year = {2020}
"""))  # braces volontairement non équilibrées

# Commit initial
subprocess.run(["git", "add", "."], cwd=WORKDIR, check=True)
subprocess.run(["git", "commit", "-m", "Initial research notes"], cwd=WORKDIR, check=True, capture_output=True)
print("✓ Workspace créé :", WORKDIR)
print("  Fichiers :", [p.name for p in WORKDIR.iterdir()])

✓ Workspace créé : /content/research_workspace
  Fichiers : ['references.bib', '.git', 'paper_notes.md']


Cellule 5 — Créer le serveur MCP custom (FastMCP)


In [ ]:
server_path = Path("/content/research_custom_server.py")
server_path.write_text(textwrap.dedent('''
    from fastmcp import FastMCP
    from typing import Dict, List
    import re

    mcp = FastMCP(name="research_ops")

    @mcp.tool
    def ping() -> str:
        """Health check tool."""
        return "pong"

    @mcp.tool
    def extract_citations(text: str) -> Dict:
        """Extract author-style citations like 'Smith et al. (2023)' or 'Jones & Lee (2022)' from text.
        Returns a dict with the list of unique citations found.
        """
        pattern = r"([A-Z][a-zA-Z]+(?:\\s+(?:et al\\.|&)\\s+[A-Z][a-zA-Z]+)?\\s*\\(?\\d{4}\\)?)"
        matches = re.findall(pattern, text)
        unique = sorted(set(m.strip().rstrip("().") for m in matches))
        return {"citations": unique, "count": len(unique)}

    @mcp.tool
    def format_bibliography(citations: List[str], style: str = "apa") -> str:
        """Format a list of raw citations into a bibliography string.
        style: 'apa' (numbered) or 'plain' (simple list).
        """
        lines = []
        for i, c in enumerate(sorted(citations), 1):
            if style == "apa":
                lines.append(f"[{i}] {c}.")
            else:
                lines.append(f"{i}. {c}")
        return "\\n".join(lines)

    @mcp.tool
    def summarize_text(text: str) -> Dict:
        """Compute summary statistics about a text block.
        Returns word_count, line_count, sentence_count, and a short preview.
        """
        words = text.split()
        sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
        preview = text.strip().split("\\n")[0][:200]
        return {
            "word_count": len(words),
            "line_count": len(text.splitlines()),
            "sentence_count": len(sentences),
            "preview": preview,
        }

    @mcp.tool
    def validate_bibtex(content: str) -> Dict:
        """Validate basic BibTeX structure. Returns entry count, types, and issues list."""
        entries = re.findall(r"@(\\w+)\\s*\\{", content)
        issues = []
        if not content.strip():
            issues.append("Empty BibTeX file.")
        open_b = content.count("{")
        close_b = content.count("}")
        if open_b != close_b:
            issues.append(f"Unbalanced braces: {open_b} '{{' vs {close_b} '}}'.")
        return {
            "entry_count": len(entries),
            "entry_types": sorted(set(entries)),
            "issues": issues,
            "valid": len(issues) == 0,
        }

    @mcp.tool
    def generate_changelog(diff_text: str) -> str:
        """Generate a human-readable changelog from a unified git diff."""
        if not diff_text.strip():
            return "No changes detected."
        added = sum(1 for l in diff_text.splitlines() if l.startswith("+") and not l.startswith("+++"))
        removed = sum(1 for l in diff_text.splitlines() if l.startswith("-") and not l.startswith("---"))
        files = sorted(set(re.findall(r"^\\+\\+\\+ b/(.+)$", diff_text, re.MULTILINE)))
        lines = [f"Changelog summary:", f"- Files changed: {len(files)}"]
        for f in files:
            lines.append(f"  • {f}")
        lines.append(f"- Lines added: {added}")
        lines.append(f"- Lines removed: {removed}")
        return "\\n".join(lines)

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''), encoding="utf-8")
print("✓ Serveur MCP custom écrit :", server_path)

✓ Serveur MCP custom écrit : /content/research_custom_server.py


Cellule 6 — Configurer les connexions MCP et charger les outils


In [ ]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
import sys
import io
from langchain_mcp_adapters.client import MultiServerMCPClient

# Patch robuste pour Google Colab : force fileno() à renvoyer un FD valide
# même si l'objet ioprocess tente de lever une exception.
def force_fileno(stream, fd):
    stream.fileno = lambda: fd
    if hasattr(stream, '_original_stdstream_copy'):
        stream._original_stdstream_copy = fd

force_fileno(sys.stdout, 1)
force_fileno(sys.stderr, 2)

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", str(WORKDIR)],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", str(WORKDIR)],
    },
    "research_ops": {
        "transport": "stdio",
        "command": "python",
        "args": [str(server_path)],
    },
}

client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
tools = asyncio.get_event_loop().run_until_complete(client.get_tools())

print(f"✓ {len(tools)} outils chargés :")
for t in tools:
    print(f"  • {t.name}")

✓ 32 outils chargés :
  • filesystem_read_file
  • filesystem_read_text_file
  • filesystem_read_media_file
  • filesystem_read_multiple_files
  • filesystem_write_file
  • filesystem_edit_file
  • filesystem_create_directory
  • filesystem_list_directory
  • filesystem_list_directory_with_sizes
  • filesystem_directory_tree
  • filesystem_move_file
  • filesystem_search_files
  • filesystem_get_file_info
  • filesystem_list_allowed_directories
  • git_git_status
  • git_git_diff_unstaged
  • git_git_diff_staged
  • git_git_diff
  • git_git_commit
  • git_git_add
  • git_git_reset
  • git_git_log
  • git_git_create_branch
  • git_git_checkout
  • git_git_show
  • git_git_branch
  • research_ops_ping
  • research_ops_extract_citations
  • research_ops_format_bibliography
  • research_ops_summarize_text
  • research_ops_validate_bibtex
  • research_ops_generate_changelog


Cellule 7 — Construire l'agent Gemini (ReAct via LangGraph)


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# Correction du nom du modèle pour l'API Google AI Studio
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-pro-latest",
    temperature=0.2,
)

SYSTEM_PROMPT = """You are a Research Assistant agent with access to three MCP servers:

1. **filesystem** — read/write/list files in the research workspace.
2. **git** — version control: status, log, diff, commit.
3. **research_ops** — extract citations, format bibliography, summarize text,
   validate BibTeX, generate changelog from a diff.

WORKFLOW GUIDELINES:
- Decide which tool to call NEXT based on the user's request.
- Do NOT call every tool blindly — plan a minimal sequence of steps.
- After reading a file, use research_ops tools to analyze it when relevant.
- When you create or modify a file, offer to commit it via git.
- Always explain your reasoning before each tool call.
- Cite which server/tool you used in your final answer.
"""

agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)
print("✓ Agent Gemini ReAct ré-initialisé avec gemini-1.5-pro-latest")

✓ Agent Gemini ReAct ré-initialisé avec gemini-1.5-pro-latest


/tmp/ipykernel_3919/3760956121.py:26: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)


Cellule 8 — Afficheur de résultats (trace lisible)


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import asyncio

async def run_query(query: str):
    print("=" * 80)
    print("👤 USER:", query)
    print("=" * 80)
    try:
        result = await agent.ainvoke({"messages": [HumanMessage(content=query)]})
        for msg in result["messages"]:
            if isinstance(msg, AIMessage) and msg.tool_calls:
                print("\n🤔 Agent thinks → calling tools:")
                for tc in msg.tool_calls:
                    args_str = str(tc["args"])
                    if len(args_str) > 200:
                        args_str = args_str[:200] + "…"
                    print(f"   🔧 {tc['name']}({args_str})")
            elif isinstance(msg, ToolMessage):
                content = msg.content if isinstance(msg.content, str) else str(msg.content)
                if len(content) > 300:
                    content = content[:300] + "…"
                print(f"   ↳ {content}")
            elif isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
                print(f"\n🤖 ASSISTANT:\n{msg.content}")
    except Exception as e:
        print(f"\n❌ Error: {e}")
    print()

# Test 1 : pipeline complet avec le nouveau compte
asyncio.get_event_loop().run_until_complete(run_query(
    "Read paper_notes.md, extract all citations from it, format them as an APA "
    "bibliography, save the result to bibliography.md, then commit the new file "
    "to git with a descriptive message."
))

👤 USER: Read paper_notes.md, extract all citations from it, format them as an APA bibliography, save the result to bibliography.md, then commit the new file to git with a descriptive message.



❌ Error: Error calling model 'gemini-1.5-pro-latest' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-pro-latest is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}



Cellule 9 — Test 2 : validation BibTeX + diagnostic


In [ ]:
asyncio.get_event_loop().run_until_complete(run_query(
    "Read references.bib and validate its BibTeX structure. If there are issues, "
    "explain them and propose a fix."
))

👤 USER: Read references.bib and validate its BibTeX structure. If there are issues, explain them and propose a fix.

❌ Error: Error calling model 'gemini-1.5-pro-latest' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-pro-latest is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}



Cellule 10 — Test 3 : résumé + historique Git


In [ ]:
asyncio.get_event_loop().run_until_complete(run_query(
    "Summarize the text content of paper_notes.md (use the summarize_text tool), "
    "then show me the git log of the repository."
))

👤 USER: Summarize the text content of paper_notes.md (use the summarize_text tool), then show me the git log of the repository.

❌ Error: Error calling model 'gemini-1.5-pro-latest' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-pro-latest is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}



Cellule 11 — Test 4 : navigation libre (l'agent décide seul)


In [ ]:
asyncio.get_event_loop().run_until_complete(run_query(
    "Give me a full report of the current research workspace: list all files, "
    "check git status, and tell me if there are any uncommitted changes."
))

👤 USER: Give me a full report of the current research workspace: list all files, check git status, and tell me if there are any uncommitted changes.

❌ Error: Error calling model 'gemini-1.5-pro-latest' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-pro-latest is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}



Cellule 12 — Nettoyage (optionnel)


In [ ]:
# Les sous-processus MCP sont automatiquement tués à la fermeture du kernel.
# Pour nettoyer manuellement le workspace :
# import shutil; shutil.rmtree(WORKDIR)
print("Done. Le workspace est conservé dans :", WORKDIR)

Done. Le workspace est conservé dans : /content/research_workspace


┌─────────────────────────────────────────────┐
│         Agent Gemini (ReAct / LangGraph)     │
│   Décide dynamiquement du prochain outil     │
└──────────────┬──────────────────────────────┘
               │ LangChain tools (via adapter)
    ┌──────────┼──────────────────┐
    ▼          ▼                   ▼
┌────────┐ ┌────────┐      ┌─────────────────┐
│ files  │ │  git   │      │ research_ops    │
│  MCP   │ │  MCP   │      │  (FastMCP)      │
│ (npx)  │ │ (pip)  │      │  custom Python  │
└────────┘ └────────┘      └─────────────────┘
   stdio      stdio              stdio

| Aspect | Détail |
|---|---|
| **3 serveurs MCP** | `filesystem` (officiel npm), `git` (officiel PyPI), `research_ops` (custom FastMCP) |
| **Transport** | `stdio` partout (sous-processus lancés par l'agent) |
| **Outils custom** | `extract_citations`, `format_bibliography`, `summarize_text`, `validate_bibtex`, `generate_changelog`, `ping` |
| **Agent** | `create_react_agent` de LangGraph → l'agent **décide** de la séquence d'appels |
| **LLM** | `gemini-2.0-flash` via `langchain-google-genai` |
| **Async** | `nest_asyncio` pour réconcilier la boucle Colab + MCP async |

